# 03 · Evidence-supervised legal DeBERTa-v3
Select a Colab GPU. Upload the matching source ZIP. Optionally provide the official archive, or review its terms and enable download. FAST_TRAIN defaults on for a two-hour T4 session. It runs one FP16 base-model trial with a 90-minute maximum training phase, then evaluates a validation-selected checkpoint. FAST_TRAIN=False restores the two validation trials and optional large model. Read validation before running Freeze. FAST evaluates the frozen checkpoint even below the accuracy target. Accuracy of 85–90% is a research target.

Evidence retrieval fits only the training contracts. Each document–hypothesis pair is an overlapping-window bag. The learning objective combines weighted document NLI and supervised evidence relevance. The full export contains the model, evidence head, retrieval weights and research calibration. Single-pair backend deployment must be validated separately.

[Koreeda and Manning](https://aclanthology.org/2021.findings-emnlp.164/) · [NLI-pretrained checkpoint](https://huggingface.co/MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli)

In [ ]:
from pathlib import Path
import os, json, subprocess, sys, time, urllib.request, zipfile

FAST_STARTED = globals().get('FAST_STARTED', time.time())
FAST_TRAIN = True
FAST_EPOCHS = 2  # Raise to 3 only if the initial T4 benchmark leaves time.
SMOKE = False
ACCEPT_CONTRACTNLI_TERMS = False  # Set True after reviewing the official terms.
MOUNT_DRIVE = False              # Only enable if you approve the Drive mount.
DRIVE_RUN_NAME = "laven-evidence-nli-seed13"
RUN_LARGE_TRIAL = False
EVALUATE_BELOW_TARGET = False  # Set only after reviewing validation and freezing the run.
ARCHIVE_PATH = Path("/content/contract-nli.zip")  # Optional pre-acquired release.
SOURCE_BUNDLE = Path("/content/laven-colab-source.zip")
ROOT = Path("/content/laven-colab")
ARCHIVE_SHA = "e03fc77bbf8b53e2976a250e81d8a294bc3d5e5fb014521e477dee9340d6287b"
DATASET_SHA = "6d825a619cf29c97faf50c63e03bbac46ee7195a1b8fe8c393c20f95646436eb"
BASE = ("MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli", "6f5cf0a2b59cabb106aca4c287eed12e357e90eb")
LARGE = ("MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli", "b3546ea6b0346eb6f8d5d68b13c7dc6d0376b3d7")

In [ ]:
from google.colab import files
from packaging.version import Version
import torch

assert torch.cuda.is_available(), "Select a GPU runtime before proceeding."
assert Version(torch.__version__.split("+")[0]) >= Version("2.6"), "Use a current Colab GPU runtime."
free, total = torch.cuda.mem_get_info()
native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
print(f"{torch.cuda.get_device_name()} · {free/2**30:.1f}/{total/2**30:.1f} GiB free · " +
      ("FP16" if FAST_TRAIN or not native_bf16 else "BF16"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.23.0",
                "sentencepiece>=0.2,<1", "safetensors>=0.5,<1", "scikit-learn>=1.5,<2",
                "scipy>=1.12,<2", "matplotlib>=3.8,<4"], check=True)

if not SOURCE_BUNDLE.is_file():
    print("Upload laven-colab-source.zip from artifacts/contractnli.")
    uploaded = files.upload()
    if SOURCE_BUNDLE.name not in uploaded:
        raise FileNotFoundError(SOURCE_BUNDLE.name)
    SOURCE_BUNDLE.write_bytes(uploaded[SOURCE_BUNDLE.name])
members = ["__init__.py", "datasets.py", "prepare_contractnli.py", "nli.py", "plots.py", "calibration.py",
           "evidence_nli.py", "evidence_retrieval.py", "diagnose_nli.py", "fast_evidence.py"]
(ROOT / "research").mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE_BUNDLE) as bundle:
    for name in members:
        target = ROOT / "research" / name
        target.write_bytes(bundle.read(f"research/{name}"))
os.chdir(ROOT)

def run(module, *args):
    subprocess.run([sys.executable, "-u", "-m", module, *map(str, args)], check=True)

In [ ]:
if not ARCHIVE_PATH.is_file() and not ACCEPT_CONTRACTNLI_TERMS:
    raise RuntimeError("Supply the authorized archive or review the terms and opt in in the configuration cell.")
if not ARCHIVE_PATH.is_file():
    run("research.datasets", "download", "contractnli", ARCHIVE_PATH,
        "--url", "https://stanfordnlp.github.io/contract-nli/resources/contract-nli.zip",
        "--sha256", ARCHIVE_SHA)

data = ROOT / "contractnli.jsonl"
run("research.prepare_contractnli", ARCHIVE_PATH, data, "--sha256", ARCHIVE_SHA)
manifest_path = data.with_suffix(".manifest.json")
manifest = json.loads(manifest_path.read_text())
assert manifest["output_sha256"] == DATASET_SHA, "Prepared dataset differs from the local comparison protocol."
run("research.plots", "dataset", manifest_path, ROOT / "dataset-figures")

In [ ]:
run_root = ROOT / "runs"
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    run_root = Path("/content/drive/MyDrive/LAVEN")
run_root = run_root / (DRIVE_RUN_NAME + ("-fast" if FAST_TRAIN else "") + ("-smoke" if SMOKE else ""))
run_root.mkdir(parents=True, exist_ok=True)
specs = [("base-lr1e5-weighted", BASE, 1e-5, .5),
         ("base-lr2e5-unweighted", BASE, 2e-5, 0.)]
if FAST_TRAIN:
    assert FAST_EPOCHS in (2, 3), "FAST_EPOCHS must be 2 or 3"
    specs = [("fast-base-lr2e5-weighted", BASE, 2e-5, .5)]
elif SMOKE:
    specs = specs[:1]
elif RUN_LARGE_TRIAL:
    assert free >= 27 * 2**30, "The large trial requires at least 27 GiB free. Use an A100-class GPU."
    specs.append(("large-lr1e5-weighted", LARGE, 1e-5, .5))

trial_paths = {}
for name, (source, revision), lr, weight in specs:
    batch = 1 if SMOKE else (2 if source == LARGE[0] else 4 if free >= 28 * 2**30 else 2 if free >= 12 * 2**30 else 1)
    common = ["--device", "cuda", "--epochs", 1 if SMOKE else (FAST_EPOCHS if FAST_TRAIN else 4), "--patience", 2,
              "--batch-size", batch, "--accumulation", 1 if SMOKE else 16 // batch,
              "--windows", 5, "--max-length", 512, "--source", source, "--revision", revision,
              "--lr", lr, "--weight-power", weight]
    if FAST_TRAIN:
        common += ["--fast", "--budget-seconds", 150 if SMOKE else 5400,
                   "--stop-at", FAST_STARTED + 95 * 60]
    elif free < 28 * 2**30 or source == LARGE[0]:
        common.append("--gradient-checkpointing")
    if SMOKE:
        common.append("--smoke")
    trial = run_root / name
    trial_paths[name] = (trial, common)
    if not (trial / "completed.json").exists():
        if (run_root / "selection.json").is_file():
            raise RuntimeError("Selection is already frozen. Additional trials after opening test are excluded.")
        resume = ["--resume"] if (trial / "last.pt").exists() else []
        run("research.evidence_nli", "train", data, trial, *common, *resume)
    result = json.loads((trial / "completed.json").read_text())
    print(name, "accuracy", result["validation"]["accuracy"],
          "macro F1", result["validation"]["macro"]["f1"], "best epoch", result["best_epoch"])
    if FAST_TRAIN:
        print("Training exposure:", result.get("training_coverage"))

In [ ]:
if SMOKE:
    print("Validation-only smoke is complete. The official test was not evaluated.")
else:
    trials = {name: json.loads((trial / "completed.json").read_text())
              for name, (trial, _) in trial_paths.items()}
    winner = max(trials, key=lambda name: (trials[name]["validation"]["macro"]["f1"],
                                           trials[name]["validation"]["accuracy"], name))
    chosen = trials[winner]["validation"]
    print("Validation winner:", winner, "accuracy:", chosen["accuracy"], "macro F1:", chosen["macro"]["f1"])
    if chosen["accuracy"] < .85 and not (EVALUATE_BELOW_TARGET or FAST_TRAIN):
        raise RuntimeError("Below 85% validation. Decide on the optional large trial or set EVALUATE_BELOW_TARGET=True before opening test.")
    selection_path = run_root / "selection.json"
    selection = {"winner": winner, "rule": "validation macro F1 then accuracy then trial name", "trials": trials}
    if selection_path.is_file() and json.loads(selection_path.read_text()) != selection:
        raise RuntimeError("Test evaluation was already frozen for another trial set.")
    selection_path.write_text(json.dumps(selection, indent=2))
    output = run_root / "final"
    if FAST_TRAIN:
        print("Minutes left in two-hour session:", round(max(0, FAST_STARTED + 7200 - time.time())/60, 1))
    if not (output / "metrics.json").exists():
        trial, flags = trial_paths[winner]
        run("research.evidence_nli", "finalize", data, output, "--trial", trial, *flags)

In [ ]:
if SMOKE:
    print("Smoke artifacts stay in the run directory. Calibration and official test were not run.")
else:
    figures = output / "figures"
    run("research.calibration", output / "predictions-calibration.csv", output / "calibration.json",
        "--test", output / "predictions-test.csv")
    run("research.plots", "nli", output, figures)
    run("research.plots", "calibration", output / "calibration.json", figures)
    export = Path("/content/laven-deberta-contractnli" + ("-fast" if FAST_TRAIN else "") + ".zip")
    export_files = [output / name for name in ("run.json", "metrics.json", "history.json", "calibration.json",
                                                "predictions-test.csv", "predictions-calibration.csv", "retriever.json")]
    export_files += list((output / "model").glob("*")) + list(figures.glob("*"))
    compression = zipfile.ZIP_STORED if FAST_TRAIN else zipfile.ZIP_DEFLATED
    with zipfile.ZipFile(export, "w", compression=compression, compresslevel=1) as bundle:
        for path in export_files:
            if path.is_file():
                bundle.write(path, path.relative_to(output))
        bundle.write(selection_path, "selection.json")
        if FAST_TRAIN:
            for name in ("fast-profile.json", "fast-coverage.json"):
                bundle.write(trial_paths[winner][0] / name, name)
        bundle.write(manifest_path, "dataset.manifest.json")
        for path in (ROOT / "dataset-figures").glob("*"):
            bundle.write(path, Path("figures") / path.name)
        bundle.writestr("labels.json", json.dumps(["Entailment","Contradiction","NotMentioned"]))
    print("Research metrics:", json.loads((output / "metrics.json").read_text())["accuracy"])
    files.download(str(export))